In [9]:
import os
import time
import wave
import serial

PORT = "COM4"
BAUDRATE = 921600
SAMPLE_RATE = 8000
CHANNELS = 1
SAMPLE_WIDTH = 2
RECORD_SECONDS = 10
SAVE_FOLDER = "records"

TOTAL_BYTES = SAMPLE_RATE * CHANNELS * SAMPLE_WIDTH * RECORD_SECONDS

def next_name(folder):
    os.makedirs(folder, exist_ok=True)
    i = 1
    while True:
        path = os.path.join(folder, f"lung_record_{i:03d}.wav")
        if not os.path.exists(path):
            return path
        i += 1

ser = serial.Serial(PORT, BAUDRATE, timeout=5)
time.sleep(2)

data = bytearray()
while len(data) < TOTAL_BYTES:
    chunk = ser.read(min(4096, TOTAL_BYTES - len(data)))
    if not chunk:
        break
    data.extend(chunk)

ser.close()

filename = next_name(SAVE_FOLDER)

with wave.open(filename, "wb") as wf:
    wf.setnchannels(CHANNELS)
    wf.setsampwidth(SAMPLE_WIDTH)
    wf.setframerate(SAMPLE_RATE)
    wf.writeframes(data)

print("Saved:", filename)
print("Bytes:", len(data), "/", TOTAL_BYTES)

Saved: records\lung_record_002.wav
Bytes: 407 / 160000


In [11]:
import os
import time
import wave
import serial

PORT = "COM4"          # عدله
BAUDRATE = 921600
SAMPLE_RATE = 8000
CHANNELS = 1
SAMPLE_WIDTH = 2
RECORD_SECONDS = 10
SAVE_FOLDER = "records"

TOTAL_BYTES = SAMPLE_RATE * CHANNELS * SAMPLE_WIDTH * RECORD_SECONDS
MAX_WAIT_SECONDS = 15

def next_name(folder):
    os.makedirs(folder, exist_ok=True)
    i = 1
    while True:
        path = os.path.join(folder, f"lung_record_{i:03d}.wav")
        if not os.path.exists(path):
            return path
        i += 1

ser = serial.Serial(PORT, BAUDRATE, timeout=0.2)

# reset ESP32
ser.setDTR(False)
time.sleep(0.2)
ser.setDTR(True)

# استنى البورد يقوم
time.sleep(3)

# امسح أي garbage قديم
ser.reset_input_buffer()

print("Recording...")

data = bytearray()
start = time.time()

while len(data) < TOTAL_BYTES and (time.time() - start) < MAX_WAIT_SECONDS:
    chunk = ser.read(min(4096, TOTAL_BYTES - len(data)))
    if chunk:
        data.extend(chunk)

ser.close()

filename = next_name(SAVE_FOLDER)

with wave.open(filename, "wb") as wf:
    wf.setnchannels(CHANNELS)
    wf.setsampwidth(SAMPLE_WIDTH)
    wf.setframerate(SAMPLE_RATE)
    wf.writeframes(data)

print("Saved:", filename)
print("Bytes:", len(data), "/", TOTAL_BYTES)

if len(data) != TOTAL_BYTES:
    print("WARNING: recording incomplete")

Recording...
Saved: records\lung_record_003.wav
Bytes: 147200 / 160000


In [13]:
import os
import time
import wave
import struct
import serial

PORT = "COM4"          # غيره لو عندك مختلف
BAUDRATE = 921600
SAMPLE_RATE = 16000
CHANNELS = 1
SAMPLE_WIDTH = 2
SAVE_FOLDER = "records"
MAGIC = b"LUNGWAV1"

def next_name(folder):
    os.makedirs(folder, exist_ok=True)
    i = 1
    while True:
        path = os.path.join(folder, f"lung_record_{i:03d}.wav")
        if not os.path.exists(path):
            return path
        i += 1

def wait_for_magic(ser, magic, timeout_sec=15):
    buf = bytearray()
    start = time.time()

    while time.time() - start < timeout_sec:
        b = ser.read(1)
        if not b:
            continue

        buf += b
        if len(buf) > len(magic):
            buf = buf[-len(magic):]

        if bytes(buf) == magic:
            return True

    return False

def read_exact(ser, n, idle_timeout_sec=5):
    data = bytearray()
    last_data_time = time.time()

    while len(data) < n:
        chunk = ser.read(min(4096, n - len(data)))
        if chunk:
            data.extend(chunk)
            last_data_time = time.time()
        else:
            if time.time() - last_data_time > idle_timeout_sec:
                break

    return bytes(data)

ser = serial.Serial(PORT, BAUDRATE, timeout=0.2)

# reset ESP32
ser.setDTR(False)
time.sleep(0.2)
ser.reset_input_buffer()
ser.setDTR(True)

print("Recording 10 seconds...")

if not wait_for_magic(ser, MAGIC, timeout_sec=15):
    ser.close()
    raise RuntimeError("No valid audio header received")

length_bytes = read_exact(ser, 4, idle_timeout_sec=3)
if len(length_bytes) != 4:
    ser.close()
    raise RuntimeError("Failed to read audio length")

total_bytes = struct.unpack("<I", length_bytes)[0]
audio_data = read_exact(ser, total_bytes, idle_timeout_sec=5)

ser.close()

filename = next_name(SAVE_FOLDER)

with wave.open(filename, "wb") as wf:
    wf.setnchannels(CHANNELS)
    wf.setsampwidth(SAMPLE_WIDTH)
    wf.setframerate(SAMPLE_RATE)
    wf.writeframes(audio_data)

print("Saved:", filename)
print("Bytes:", len(audio_data), "/", total_bytes)

if len(audio_data) != total_bytes:
    print("WARNING: recording incomplete")

Recording 10 seconds...
Saved: records\lung_record_005.wav
Bytes: 320000 / 320000
